In [91]:
import glob
import pandas as pd
import altair as alt

In [92]:
parquet_files = glob.glob("ensemble_outputs/*.parquet")

# 3. Read each file and add the filename column using a list comprehension
df_list = []
for file_path in parquet_files:
    df = pd.read_parquet(file_path)
    df["file_name"] = file_path.split('/')[-1].split('.parquet')[0]  # Use file_path.name for 'file.parquet' or str(file_path) for full path
    df_list.append(df)

# 4. Concatenate all DataFrames into one large DataFrame
df = pd.concat(df_list)

In [93]:
df = df[df.index > '20251201']
df['forecast'] = df['file_name'].apply(lambda s: s.split('_')[2])
df['ensemble'] = df['file_name'].apply(lambda s: s.split('_')[3])

In [94]:
alt.data_transformers.enable('json')

DataTransformerRegistry.enable('json')

In [95]:
alt.Chart(
    df.reset_index()
).mark_line().encode(
    alt.X('datetime:T'),
    alt.Y('sqin:Q'),
    alt.Detail('ensemble'),
    alt.Color('forecast:N')
) .properties(width=500)

alt.Chart(...)

In [96]:
df.head()

,sqin,file_name,forecast,ensemble
datetime,,,,
2025-12-01 06:00:00,982.495789,streamflow_HHDW1_mefp_2011,mefp,2011
2025-12-01 12:00:00,943.080065,streamflow_HHDW1_mefp_2011,mefp,2011
2025-12-01 18:00:00,906.773079,streamflow_HHDW1_mefp_2011,mefp,2011
2025-12-02 00:00:00,872.941999,streamflow_HHDW1_mefp_2011,mefp,2011
2025-12-02 06:00:00,842.255111,streamflow_HHDW1_mefp_2011,mefp,2011


In [97]:
percentiles = (
    df.reset_index()
    .groupby(["datetime", "forecast"])["sqin"]
    .quantile([0.05, 0.25, 0.50, 0.75, 0.95])
    .unstack()
    .rename(columns={
        0.05: "p05",
        0.25: "p25",
        0.50: "p50",
        0.75: "p75",
        0.95: "p95",
    })
    .reset_index()
)
percentiles

,datetime,forecast,p05,p25,p50,p75,p95
0,2025-12-01 06:00:00,aorc,973.184070,973.184070,973.184070,973.184070,973.184070
1,2025-12-01 06:00:00,mefp,982.267998,983.543898,984.805722,986.263719,993.671840
2,2025-12-01 06:00:00,westwrf,991.842754,992.258281,992.647151,993.074331,993.843718
3,2025-12-01 12:00:00,aorc,934.734330,934.734330,934.734330,934.734330,934.734330
4,2025-12-01 12:00:00,mefp,942.892579,944.156067,945.271967,946.756699,953.766247
...,...,...,...,...,...,...,...
237,2025-12-30 18:00:00,aorc,1133.959441,1133.959441,1133.959441,1133.959441,1133.959441
238,2025-12-31 00:00:00,aorc,1096.012036,1096.012036,1096.012036,1096.012036,1096.012036
239,2025-12-31 06:00:00,aorc,1060.454115,1060.454115,1060.454115,1060.454115,1060.454115
240,2025-12-31 12:00:00,aorc,1031.807963,1031.807963,1031.807963,1031.807963,1031.807963


In [98]:
observed_flow = pd.read_csv("/Users/elischwat/Development/nwsrfs-hydro-models/nwsrfs_py/nwsrfs_py/data/HHDW1_archive/flow_daily_HHDW1.csv")
observed_flow = observed_flow.assign(
    datetime = pd.to_datetime(observed_flow[['year', 'month', 'day']]),
    forecast = 'Measured'
).drop(
    columns = ['year', 'month', 'day']
)[['datetime', 'flow_cfs', 'forecast']].rename(
    columns={'flow_cfs':'p50'}
)
observed_flow

,datetime,p50,forecast
0,1979-10-01,142.979061,Measured
1,1979-10-02,141.114568,Measured
2,1979-10-03,139.798332,Measured
3,1979-10-04,133.460268,Measured
4,1979-10-05,133.598740,Measured
...,...,...,...
15702,2022-09-27,142.119348,Measured
15703,2022-09-28,133.817595,Measured
15704,2022-09-29,149.329792,Measured
15705,2022-09-30,153.478360,Measured


In [99]:
observed_flow = pd.read_csv("/Users/elischwat/Downloads/DETO3I_dataquery.csv")
observed_flow = observed_flow.assign(
    datetime = pd.to_datetime(observed_flow['Date Time']),
    p50 = observed_flow.iloc[:,1],
    forecast = 'Measured'
).iloc[:,2:]
observed_flow

,datetime,p50,forecast
0,2025-12-06 23:00:00,2982.616,Measured
1,2025-12-07 00:00:00,3055.760,Measured
2,2025-12-07 01:00:00,3132.231,Measured
3,2025-12-07 02:00:00,3230.276,Measured
4,2025-12-07 03:00:00,3181.370,Measured
...,...,...,...
188,2025-12-14 19:00:00,2009.067,Measured
189,2025-12-14 20:00:00,1977.707,Measured
190,2025-12-14 21:00:00,1926.441,Measured
191,2025-12-14 22:00:00,1934.082,Measured


In [100]:
base_chart = alt.Chart().encode(
    alt.X('datetime:T').title('December 2025'),
    alt.Color('forecast:N')
    
)
chart_05_to_25 = base_chart.mark_area(opacity=0.25).encode(alt.Y('p05'), alt.Y2('p25'))
chart_25_to_50 = base_chart.mark_area(opacity=0.62).encode(alt.Y('p25'), alt.Y2('p50'))
chart_50_to_75 = base_chart.mark_area(opacity=0.62).encode(alt.Y('p50'), alt.Y2('p75'))
chart_75_to_95 = base_chart.mark_area(opacity=0.25).encode(alt.Y('p75'), alt.Y2('p95'))

chart_50_line = base_chart.mark_line().encode(
    alt.Y('p50').title(['Streamflow (cfs)', '(5, 25, 50, 75, & 95 percentiles )']),
)

aorc_line_chart = alt.Chart().transform_filter( 
        alt.datum.source == 'aorc'
    ).mark_line().encode(
        alt.X('datetime:T'),
        alt.Y('sqin'),
        alt.Color('forecast:N')
    )
# Create dummy data for now
# observed = pd.concat([
#     observed_flow[observed_flow.datetime > '20251201'],
#     pd.DataFrame({'datetime':[pd.to_datetime('20251201')], 'p50': [0], 'forecast': ['Measured']})
# ])

plotting_df = pd.concat([
    percentiles,
    observed_flow
])

alt.layer(
    # aorc_line_chart,
    chart_05_to_25,
    chart_25_to_50,
    chart_50_to_75,
    chart_75_to_95, 
    chart_50_line,
    data = plotting_df[plotting_df.datetime < "20251215"],
).properties(width=500)

alt.LayerChart(...)

alt.LayerChart(...)